In [149]:
import pandas as pd
import numpy as np

# Data Preprocessing 

In [150]:
mvp_df = pd.read_csv("data/nba_mvp_voting_2010_to_2025.csv")
standings_df = pd.read_csv("data/nba_standings_2010_to_2025.csv")

In [151]:
# Remove Columns not needed for analysis and rename
mvp_df = mvp_df.drop(columns=["Player", "Rank", "Age", "First", "Share", "Pts Max", "WS/48"], errors='ignore')
mvp_df = mvp_df.rename(columns={"Tm": "Team"})
standings_df = standings_df.drop(columns=["W", "L", "PS/G", "PA/G", "SRS", "GB", "Conference"], errors='ignore')


In [152]:
# Map old team abbreviations to new ones
team_abbrev = {
    "NOH": "NOP",
    "CHA": "CHO",
}

mvp_df["Team"] = mvp_df["Team"].replace(team_abbrev)
mvp_df


,Season,Team,Pts Won,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS
0,2009-10,CLE,1205,76,39.0,29.7,7.3,8.6,1.6,1.0,0.503,0.333,0.767,18.5
1,2009-10,OKC,609,82,39.5,30.1,7.6,2.8,1.4,1.0,0.476,0.365,0.900,16.1
2,2009-10,LAL,599,73,38.8,27.0,5.4,5.0,1.5,0.3,0.456,0.329,0.811,9.4
3,2009-10,ORL,478,82,34.7,18.3,13.2,1.8,0.9,2.8,0.612,0.000,0.592,13.2
4,2009-10,MIA,119,77,36.3,26.6,4.8,6.5,1.8,1.1,0.476,0.300,0.761,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-25,MIN,12,79,36.3,27.6,5.7,4.5,1.2,0.6,0.447,0.395,0.837,8.4
203,2024-25,GSW,2,70,32.2,24.5,4.4,6.0,1.1,0.4,0.448,0.397,0.933,7.9
204,2024-25,NYK,1,65,35.4,26.0,2.9,7.3,0.9,0.1,0.488,0.383,0.821,8.3
205,2024-25,LAC,1,79,35.3,22.8,5.8,8.7,1.5,0.7,0.410,0.352,0.874,8.3


In [153]:
# Convert games to % of Games Played - to account for shortened seasons and enable the model to generalize better
def get_season_games(season):
    year = int(season.split("-")[1])

    if year == 12:
        return 66
    elif year == 20:
        return 67
    elif year == 21:
        return 72
    else:
        return 82

mvp_df["G"] = mvp_df["G"] / mvp_df["Season"].apply(get_season_games)

In [154]:
# Normalize stats by year to account for stats inflation
stats_columns = ["PTS", "TRB", "AST", "STL", "BLK", "FG%", "3P%", "FT%"]
for col in stats_columns:
    mvp_df[col] = mvp_df.groupby("Season")[col].transform(
        lambda x: (x - x.mean()) / x.std()
    )

mvp_df

,Season,Team,Pts Won,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS
0,2009-10,CLE,1205,0.926829,39.0,1.440440,0.308217,1.190652,1.059750,0.392503,0.382933,0.087984,-0.586038,18.5
1,2009-10,OKC,609,1.000000,39.5,1.526522,0.411723,-0.713516,0.562992,0.392503,-0.148190,0.383317,0.962499,16.1
2,2009-10,LAL,599,0.890244,38.8,0.859386,-0.347319,0.008755,0.811371,-0.612688,-0.541614,0.051068,-0.073740,9.4
3,2009-10,ORL,478,1.000000,34.7,-1.012899,2.343830,-1.041821,-0.678903,2.977279,2.527096,-2.985318,-2.623586,13.2
4,2009-10,MIA,119,0.939024,36.3,0.773304,-0.554331,0.501212,1.556508,0.536102,-0.148190,-0.216577,-0.655897,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-25,MIN,12,0.963415,36.3,0.394449,-0.446564,-1.108340,0.000000,-0.218888,-0.777047,0.657426,0.277546,8.4
203,2024-25,GSW,2,0.853659,32.2,-0.429905,-0.871345,-0.372173,-0.319142,-0.696463,-0.760367,0.698302,1.444452,7.9
204,2024-25,NYK,1,0.792683,35.4,-0.031024,-1.361477,0.265838,-0.957427,-1.412826,-0.093134,0.412168,0.083061,8.3
205,2024-25,LAC,1,0.963415,35.3,-0.881970,-0.413889,0.952927,0.957427,0.019899,-1.394237,-0.221413,0.727291,8.3


In [155]:
# Calculate percent of pts won for each player for each season
mvp_df["Pts Won %"] = mvp_df.groupby("Season")["Pts Won"].transform(
    lambda x: x / x.sum()
)

mvp_df.drop(columns=["Pts Won"], inplace=True)
mvp_df

,Season,Team,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,Pts Won %
0,2009-10,CLE,0.926829,39.0,1.440440,0.308217,1.190652,1.059750,0.392503,0.382933,0.087984,-0.586038,18.5,0.376798
1,2009-10,OKC,1.000000,39.5,1.526522,0.411723,-0.713516,0.562992,0.392503,-0.148190,0.383317,0.962499,16.1,0.190432
2,2009-10,LAL,0.890244,38.8,0.859386,-0.347319,0.008755,0.811371,-0.612688,-0.541614,0.051068,-0.073740,9.4,0.187305
3,2009-10,ORL,1.000000,34.7,-1.012899,2.343830,-1.041821,-0.678903,2.977279,2.527096,-2.985318,-2.623586,13.2,0.149468
4,2009-10,MIA,0.939024,36.3,0.773304,-0.554331,0.501212,1.556508,0.536102,-0.148190,-0.216577,-0.655897,13.0,0.037211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,2024-25,MIN,0.963415,36.3,0.394449,-0.446564,-1.108340,0.000000,-0.218888,-0.777047,0.657426,0.277546,8.4,0.004615
203,2024-25,GSW,0.853659,32.2,-0.429905,-0.871345,-0.372173,-0.319142,-0.696463,-0.760367,0.698302,1.444452,7.9,0.000769
204,2024-25,NYK,0.792683,35.4,-0.031024,-1.361477,0.265838,-0.957427,-1.412826,-0.093134,0.412168,0.083061,8.3,0.000385
205,2024-25,LAC,0.963415,35.3,-0.881970,-0.413889,0.952927,0.957427,0.019899,-1.394237,-0.221413,0.727291,8.3,0.000385


In [156]:
# Map team full name to abbreviations in standings_df
team_full_to_abbrev = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "New Jersey Nets": "BRK",
    "Charlotte Hornets": "CHO",
    "Charlotte Bobcats": "CHO",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New Orleans Hornets": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

standings_df['Team'] = standings_df['Team'].map(team_full_to_abbrev)
standings_df

,Season,Team,W/L%
0,2009-10,BOS,0.610
1,2009-10,TOR,0.488
2,2009-10,NYK,0.354
3,2009-10,PHI,0.329
4,2009-10,BRK,0.146
...,...,...,...
475,2024-25,PHO,0.439
476,2024-25,POR,0.439
477,2024-25,SAS,0.415
478,2024-25,NOP,0.256


In [157]:
# Add W/L% to MVP data and drop season and team data now
df = pd.merge(mvp_df, standings_df, on=["Season", "Team"], how='left')
df = df.drop(columns=["Season", "Team"], errors='ignore')
df

,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,Pts Won %,W/L%
0,0.926829,39.0,1.440440,0.308217,1.190652,1.059750,0.392503,0.382933,0.087984,-0.586038,18.5,0.376798,0.744
1,1.000000,39.5,1.526522,0.411723,-0.713516,0.562992,0.392503,-0.148190,0.383317,0.962499,16.1,0.190432,0.610
2,0.890244,38.8,0.859386,-0.347319,0.008755,0.811371,-0.612688,-0.541614,0.051068,-0.073740,9.4,0.187305,0.695
3,1.000000,34.7,-1.012899,2.343830,-1.041821,-0.678903,2.977279,2.527096,-2.985318,-2.623586,13.2,0.149468,0.720
4,0.939024,36.3,0.773304,-0.554331,0.501212,1.556508,0.536102,-0.148190,-0.216577,-0.655897,13.0,0.037211,0.573
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,0.963415,36.3,0.394449,-0.446564,-1.108340,0.000000,-0.218888,-0.777047,0.657426,0.277546,8.4,0.004615,0.598
203,0.853659,32.2,-0.429905,-0.871345,-0.372173,-0.319142,-0.696463,-0.760367,0.698302,1.444452,7.9,0.000769,0.585
204,0.792683,35.4,-0.031024,-1.361477,0.265838,-0.957427,-1.412826,-0.093134,0.412168,0.083061,8.3,0.000385,0.622
205,0.963415,35.3,-0.881970,-0.413889,0.952927,0.957427,0.019899,-1.394237,-0.221413,0.727291,8.3,0.000385,0.610


In [158]:
# Create X and Y datasets and convert them to numpy arrays for model training
X_train = df.drop(columns=["Pts Won %"]).fillna(0).to_numpy()
Y_train = df[["Pts Won %"]].to_numpy().ravel()

In [159]:
# Preprocess test data (2025 - 2026)
test_df = pd.read_csv("data/nba_per_game_stats_2025_to_2026.csv")
test_df = test_df[test_df["Player"] != "League Average"]
test_df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards
0,1.0,Luka Dončić,26.0,LAL,PG,13.0,13.0,37.2,10.8,23.1,...,0.8,7.9,8.8,9.2,1.9,0.5,4.2,2.7,35.2,NaN
1,2.0,Shai Gilgeous-Alexander,27.0,OKC,PG,18.0,18.0,33.1,10.8,19.9,...,0.6,4.3,4.9,6.6,1.5,0.8,1.8,1.8,32.2,NaN
2,3.0,Tyrese Maxey,25.0,PHI,PG,17.0,17.0,39.9,10.8,22.9,...,0.3,4.1,4.4,7.5,1.6,0.8,2.7,2.4,32.2,NaN
3,4.0,Giannis Antetokounmpo,31.0,MIL,PF,13.0,13.0,31.8,12.0,19.1,...,3.7,7.2,10.8,6.8,0.9,1.2,3.5,2.6,31.2,NaN
4,5.0,Donovan Mitchell,29.0,CLE,SG,17.0,17.0,34.0,10.2,20.4,...,0.8,3.9,4.8,5.5,1.4,0.4,3.2,2.2,29.9,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478,477.0,Joe Ingles,38.0,MIN,SF,6.0,0.0,3.8,0.0,0.3,...,0.0,0.0,0.0,1.0,0.2,0.0,0.0,0.3,0.0,NaN
479,478.0,Trey Jemison,26.0,NYK,C,2.0,0.0,5.0,0.0,0.0,...,0.5,0.5,1.0,0.0,0.0,0.0,0.5,1.0,0.0,NaN
480,479.0,Christian Koloko,25.0,LAL,C,2.0,0.0,3.0,0.0,0.0,...,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.0,NaN
481,480.0,Chris Mañon,24.0,LAL,SG,2.0,0.0,3.5,0.0,0.5,...,0.0,1.0,1.0,0.0,0.0,0.5,0.0,1.0,0.0,NaN


In [160]:
# Merge with advanced stats for win shares
advanced_stats_df = pd.read_csv("data/nba_advanced_stats_2025_to_2026.csv")
test_df = pd.merge(test_df, advanced_stats_df[["Player", "WS"]], on="Player", how="left")
test_df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,WS
0,1.0,Luka Dončić,26.0,LAL,PG,13.0,13.0,37.2,10.8,23.1,...,7.9,8.8,9.2,1.9,0.5,4.2,2.7,35.2,NaN,2.4
1,2.0,Shai Gilgeous-Alexander,27.0,OKC,PG,18.0,18.0,33.1,10.8,19.9,...,4.3,4.9,6.6,1.5,0.8,1.8,1.8,32.2,NaN,5.2
2,3.0,Tyrese Maxey,25.0,PHI,PG,17.0,17.0,39.9,10.8,22.9,...,4.1,4.4,7.5,1.6,0.8,2.7,2.4,32.2,NaN,2.8
3,4.0,Giannis Antetokounmpo,31.0,MIL,PF,13.0,13.0,31.8,12.0,19.1,...,7.2,10.8,6.8,0.9,1.2,3.5,2.6,31.2,NaN,2.3
4,5.0,Donovan Mitchell,29.0,CLE,SG,17.0,17.0,34.0,10.2,20.4,...,3.9,4.8,5.5,1.4,0.4,3.2,2.2,29.9,NaN,2.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,477.0,Joe Ingles,38.0,MIN,SF,6.0,0.0,3.8,0.0,0.3,...,0.0,0.0,1.0,0.2,0.0,0.0,0.3,0.0,NaN,0.0
485,478.0,Trey Jemison,26.0,NYK,C,2.0,0.0,5.0,0.0,0.0,...,0.5,1.0,0.0,0.0,0.0,0.5,1.0,0.0,NaN,0.0
486,479.0,Christian Koloko,25.0,LAL,C,2.0,0.0,3.0,0.0,0.0,...,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.0,NaN,0.0
487,480.0,Chris Mañon,24.0,LAL,SG,2.0,0.0,3.5,0.0,0.5,...,1.0,1.0,0.0,0.0,0.5,0.0,1.0,0.0,NaN,0.0


In [161]:
# Preprocess current 2025-2026 team standings 
test_standings_df = pd.read_csv("data/nba_standings_2025_to_2026.csv")
test_standings_df = test_standings_df[["Team", "W/L%"]]
test_standings_df['Team'] = test_standings_df['Team'].map(team_full_to_abbrev)
test_standings_df

,Team,W/L%
0,DET,0.882
1,TOR,0.722
2,MIA,0.667
3,CLE,0.632
4,NYK,0.625
5,ATL,0.579
6,ORL,0.579
7,CHI,0.529
8,PHI,0.529
9,BOS,0.529


In [162]:
# Merge test data with standings
test_df = pd.merge(test_df, test_standings_df, on="Team", how="left")
test_df

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,WS,W/L%
0,1.0,Luka Dončić,26.0,LAL,PG,13.0,13.0,37.2,10.8,23.1,...,8.8,9.2,1.9,0.5,4.2,2.7,35.2,NaN,2.4,0.765
1,2.0,Shai Gilgeous-Alexander,27.0,OKC,PG,18.0,18.0,33.1,10.8,19.9,...,4.9,6.6,1.5,0.8,1.8,1.8,32.2,NaN,5.2,0.944
2,3.0,Tyrese Maxey,25.0,PHI,PG,17.0,17.0,39.9,10.8,22.9,...,4.4,7.5,1.6,0.8,2.7,2.4,32.2,NaN,2.8,0.529
3,4.0,Giannis Antetokounmpo,31.0,MIL,PF,13.0,13.0,31.8,12.0,19.1,...,10.8,6.8,0.9,1.2,3.5,2.6,31.2,NaN,2.3,0.444
4,5.0,Donovan Mitchell,29.0,CLE,SG,17.0,17.0,34.0,10.2,20.4,...,4.8,5.5,1.4,0.4,3.2,2.2,29.9,NaN,2.3,0.632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,477.0,Joe Ingles,38.0,MIN,SF,6.0,0.0,3.8,0.0,0.3,...,0.0,1.0,0.2,0.0,0.0,0.3,0.0,NaN,0.0,0.588
485,478.0,Trey Jemison,26.0,NYK,C,2.0,0.0,5.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.5,1.0,0.0,NaN,0.0,0.625
486,479.0,Christian Koloko,25.0,LAL,C,2.0,0.0,3.0,0.0,0.0,...,0.5,0.0,0.0,0.0,0.0,0.5,0.0,NaN,0.0,0.765
487,480.0,Chris Mañon,24.0,LAL,SG,2.0,0.0,3.5,0.0,0.5,...,1.0,0.0,0.0,0.5,0.0,1.0,0.0,NaN,0.0,0.765


In [163]:
# Drop unnecessary columns and rename
test_df = test_df[["G", "MP", "PTS", "TRB", "AST", "STL", "BLK", "FG%", "3P%", "FT%", "WS", "W/L%"]]
test_df

,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,W/L%
0,13.0,37.2,35.2,8.8,9.2,1.9,0.5,0.470,0.333,0.789,2.4,0.765
1,18.0,33.1,32.2,4.9,6.6,1.5,0.8,0.543,0.412,0.898,5.2,0.944
2,17.0,39.9,32.2,4.4,7.5,1.6,0.8,0.470,0.409,0.878,2.8,0.529
3,13.0,31.8,31.2,10.8,6.8,0.9,1.2,0.629,0.500,0.636,2.3,0.444
4,17.0,34.0,29.9,4.8,5.5,1.4,0.4,0.503,0.383,0.832,2.3,0.632
...,...,...,...,...,...,...,...,...,...,...,...,...
484,6.0,3.8,0.0,0.0,1.0,0.2,0.0,0.000,0.000,NaN,0.0,0.588
485,2.0,5.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.625
486,2.0,3.0,0.0,0.5,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.765
487,2.0,3.5,0.0,1.0,0.0,0.0,0.5,0.000,0.000,NaN,0.0,0.765


In [164]:
test_df = test_df.fillna(0)

# Normalize test data stats given average in that year
test_df = test_df.transform(
    lambda x: (x - x.mean()) / x.std()
)

test_df

,G,MP,PTS,TRB,AST,STL,BLK,FG%,3P%,FT%,WS,W/L%
0,0.195996,1.793326,3.583640,2.088473,3.639194,2.404471,0.275779,0.080157,0.184182,0.381850,2.580461,1.180470
1,1.128311,1.386156,3.169947,0.558873,2.308180,1.612028,0.967347,0.608666,0.611285,0.778416,6.527693,1.949060
2,0.941848,2.061463,3.169947,0.362770,2.768916,1.810139,0.967347,0.080157,0.595066,0.705652,3.144352,0.167133
3,0.195996,1.257053,3.032050,2.872884,2.410566,0.423365,1.889439,1.231294,1.087046,-0.174798,2.439489,-0.197840
4,0.941848,1.475535,2.852784,0.519652,1.745059,1.413918,0.045256,0.319072,0.454500,0.538293,2.439489,0.609395
...,...,...,...,...,...,...,...,...,...,...,...,...
484,-1.109245,-1.523626,-1.270348,-1.362934,-0.558620,-0.963409,-0.876835,-3.322576,-1.616141,-2.488709,-0.802881,0.420468
485,-1.855097,-1.404454,-1.270348,-0.970728,-1.070548,-1.359630,-0.876835,-3.322576,-1.616141,-2.488709,-0.802881,0.579338
486,-1.855097,-1.603074,-1.270348,-1.166831,-1.070548,-1.359630,-0.876835,-3.322576,-1.616141,-2.488709,-0.802881,1.180470
487,-1.855097,-1.553419,-1.270348,-0.970728,-1.070548,-1.359630,0.275779,-3.322576,-1.616141,-2.488709,-0.802881,1.180470


In [165]:
X_test = test_df.to_numpy()
X_test

array([[ 0.19599581,  1.79332635,  3.58363957, ...,  0.38184981,
         2.58046125,  1.18047017],
       [ 1.12831053,  1.38615558,  3.16994746, ...,  0.7784162 ,
         6.52769344,  1.9490604 ],
       [ 0.94184758,  2.06146319,  3.16994746, ...,  0.70565172,
         3.14435156,  0.16713332],
       ...,
       [-1.85509656, -1.60307368, -1.27034777, ..., -2.48870872,
        -0.80288064,  1.18047017],
       [-1.85509656, -1.55341871, -1.27034777, ..., -2.48870872,
        -0.80288064,  1.18047017],
       [-1.85509656, -1.10652397, -1.27034777, ..., -2.48870872,
        -0.94385321, -1.59761856]], shape=(489, 12))

# Model Training

In [166]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [167]:
def ridge_cv_tuning(X, y, alphas, k=10):
    """
    Performs K-Fold Cross-Validation to find the best alpha for Ridge Regression.
    
    Parameters:
    - X: Feature matrix (numpy array or pandas DataFrame)
    - y: Target vector (numpy array or pandas Series)
    - alphas: List of alpha values (hyperparameters) to test
    - k: Number of folds for cross-validation
    
    Returns:
    - best_alpha: The alpha value with the lowest MSE
    - results: A dictionary containing the MSE scores for every alpha
    """
    
    # Ensure inputs are numpy arrays for easier indexing
    X_data = X.values if isinstance(X, pd.DataFrame) else np.array(X)
    y_data = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else np.array(y)
    
    # Initialize K-Fold
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    # Store average MSE for each alpha
    mse_scores = []
    
    print(f"Starting {k}-Fold Cross-Validation on {len(alphas)} hyperparameters...\n")
    
    for alpha in alphas:
        fold_mses = []
        
        # Split data into K folds
        for train_index, val_index in kf.split(X_data):
            # 1. Split data
            X_train, X_val = X_data[train_index], X_data[val_index]
            y_train, y_val = y_data[train_index], y_data[val_index]
            
            # 2. Initialize and Train Ridge Model
            model = Ridge(alpha=alpha)
            model.fit(X_train, y_train)
            
            # 3. Predict on Validation set
            y_pred = model.predict(X_val)
            
            # 4. Calculate MSE for this fold
            fold_mse = mean_squared_error(y_val, y_pred)
            fold_mses.append(fold_mse)
        
        # Average the MSE across all folds for this specific alpha
        avg_mse = np.mean(fold_mses)
        mse_scores.append(avg_mse)

    # Find the index of the lowest MSE
    best_idx = np.argmin(mse_scores)
    best_alpha = alphas[best_idx]
    lowest_mse = mse_scores[best_idx]
    
    return best_alpha, lowest_mse, mse_scores

In [168]:
alpha_list = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
best_param, best_score, all_scores = ridge_cv_tuning(X_train, Y_train, alpha_list, k=10)
best_param, best_score, all_scores

Starting 10-Fold Cross-Validation on 7 hyperparameters...



(0.1,
 np.float64(0.004714910605544363),
 [np.float64(0.004729863285590543),
  np.float64(0.004727870124720815),
  np.float64(0.004714910605544363),
  np.float64(0.004733872179726483),
  np.float64(0.004799791440390135),
  np.float64(0.005069936711285144),
  np.float64(0.005162039303303249)])

In [169]:
Y_pred = Ridge(alpha=best_param).fit(X_train, Y_train).predict(X_test)
Y_pred

array([ 8.15626564e-01,  1.05993786e+00,  4.62634195e-01,  3.48513651e-01,
        5.57356749e-01,  9.28927023e-01,  3.39645113e-01,  6.04299853e-02,
        4.67853154e-01,  3.76258646e-01,  1.03980506e-01,  6.98304016e-01,
        3.99115386e-01,  8.57614603e-01,  4.94106597e-01,  5.58897121e-01,
        4.74754741e-01,  2.64001206e-01, -1.76219786e-01,  5.52970013e-01,
       -3.69926198e-01, -1.32591711e-01,  4.80756336e-01,  3.65592127e-01,
        1.66244462e-01,  4.38477112e-01,  4.08678767e-02,  6.53823867e-01,
       -9.36144300e-02,  4.41904841e-01,  1.46442042e-01,  5.03542168e-01,
        6.53962442e-01, -2.43298305e-01, -7.30098824e-02,  3.32365651e-01,
        3.39082156e-01,  5.77391315e-01,  4.87888935e-01, -3.49705425e-01,
       -9.98033915e-02, -1.53360431e-01,  3.84842815e-01, -9.85820979e-02,
        8.06476630e-01, -1.40926851e-01,  3.66760837e-01,  1.31708978e-01,
        1.52466901e-01,  5.45997211e-01, -1.03671473e-01,  4.17858731e-01,
        6.18220592e-01,  